# ML-03 — Frame Your Lane as an ML Task

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjalanano-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am framing **Lane 2: Refresh / Content Opportunity Scoring** as a **Ranking and Scoring** task.

While we can train a binary classifier to estimate the probability that a page will decline in traffic, the ultimate product deliverable is a sorted queue of pages ordered by priority. The model needs to produce a continuous score representing the expected value or urgency of a refresh (combining the probability of decline with the scale of the search demand/impressions). This allows editors to review pages starting from the highest-priority opportunities down to their team's capacity limit.

In [1]:
# Show the distribution of trend_direction to understand the ranking task
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print("Trend direction counts:")
print(df['trend_direction'].value_counts())
print("\nTrend direction percentage:")
print(df['trend_direction'].value_counts(normalize=True) * 100)


Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend direction percentage:
trend_direction
down      54.206667
stable    19.873333
up        14.626667
new        7.453333
flat       3.840000
Name: proportion, dtype: float64


## 2. Target or proxy

Our target is **observed page traffic decline** in a future window, using a proxy in the starter dataset and an observed target in the warehouse:

- **Starter Dataset Proxy**: The binary target is `is_declining_label`, defined as `trend_direction == 'down'`. This is a proxy because it measures decline based on the current window (last 30 days vs. previous 30 days).
- **Warehouse Target**: The true target will be defined as an observed traffic decline (e.g., $\ge 20\%$ drop in GSC search impressions) measured in a future 30-day window, using features constructed entirely from a prior 90-day feature window. This ensures a clean temporal separation and prevents feature leakage.
- **Observation vs. Rule**: The target is an **observed outcome** in the search data (actual traffic drop), not a human-defined rule (like health scores or editor flags). This ensures the model learns real search performance changes rather than reproducing an existing algorithm.

In [2]:
# Show how the proxy target is defined and the impact of demand filtering
df['target_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"Total declining pages (proxy target): {df['target_declining'].sum()} ({df['target_declining'].mean() * 100:.1f}%)")

# Filter for pages with sufficient search demand (impressions >= 100) to rule out noise
df['target_with_demand'] = ((df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)).astype(int)
print(f"High-value priority targets (declining + >= 100 impressions): {df['target_with_demand'].sum()} ({df['target_with_demand'].mean() * 100:.1f}%)")


Total declining pages (proxy target): 16262 (54.2%)
High-value priority targets (declining + >= 100 impressions): 13152 (43.8%)


## 3. Success metric

The primary success metric for this ranking task is **Precision@K (specifically Precision@50)**, supported by **Average Precision (AP)**:

- **Precision@K**: Measures the proportion of correctly flagged declining pages within the top $K$ items of the ranked queue. We pick $K = 50$ because it aligns with a realistic weekly manual review capacity for an editorial team.
- **Why this metric**: An editor will start at the top of the list and work down. We want to maximize the probability that every page they inspect actually needs a refresh. Normal classification accuracy is irrelevant here because the list is never consumed as an unordered set; ranking quality at the head of the queue is what matters.
- **What means 'good'**: The base rate of decline in the dataset is $54.2\%$. A random ranking would yield a Precision@50 of $54.2\%$. A 'good' system should achieve a **Precision@50 of $\ge 75\%$**, meaning at least 37-38 of the top 50 suggested pages are true opportunities.

In [3]:
# Compute the base rate to establish the random guessing baseline
base_rate = (df['trend_direction'] == 'down').mean()
print(f"Base rate of decline: {base_rate:.4f}")
print(f"Expected Precision@50 of a random queue: {base_rate * 100:.1f}%")


Base rate of decline: 0.5421
Expected Precision@50 of a random queue: 54.2%


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one individual page (content item) for a specific client** over a trailing 90-day window. Each row represents a single page with its associated metadata and aggregated search and engagement performance metrics. Below is a sample showing the columns that represent our unit of analysis.

In [4]:
# Display a sample of the unit of analysis (one row = one page)
sample_cols = [
    'client_id', 
    'content_id', 
    'content_type', 
    'impressions_90d', 
    'clicks_90d', 
    'ctr', 
    'avg_position', 
    'days_since_last_update', 
    'trend_direction'
]
print(df[sample_cols].head().to_string(index=False))


        client_id           content_id    content_type  impressions_90d  clicks_90d  ctr  avg_position  days_since_last_update trend_direction
client_f369cb89fc content_304f48230142 keyword article             3803          29 0.76          10.6                      20            down
client_4e07408562 content_a1fb4e703a9e keyword article            15320           7 0.05          20.3                      25            down
client_7f2253d7e2 content_9aa793d4d895 keyword article            12581          11 0.09          36.5                      20            down
client_19581e27de content_331d6c4de07b keyword article            11751          58 0.49           6.2                      22          stable
client_3fdba35f04 content_d99b7a2d90ca keyword article            19140          24 0.13          44.0                      14            down


## 5. Why ML beats a fixed rule here

A simple rule-based approach like "refresh any page older than 180 days" or "refresh any page in position 11-20" is highly inefficient because:

1. **Staleness is not decay**: A page may be old but still maintain stable traffic due to evergreen content.
2. **Decline without volume is noise**: Small fluctuations on low-volume pages look like severe percentage drops but have zero business impact.
3. **Tangled Signals**: High-value opportunities lie in the non-linear interaction of signals: e.g., a page with moderate position decline, high search volume, low GA4 scroll rate, and high age. Writing an if-statement to handle all these thresholds (impressions, positions, CTRs, engagement rates, content types) for dozens of clients with different base rates is too complex and brittle. ML learns this multi-dimensional decision boundary automatically from the historical panel data.

In [5]:
# Analyze how simple rules over-capture or under-target
stale_threshold = 180
stale_mask = df['days_since_last_update'] >= stale_threshold
declining_mask = df['trend_direction'] == 'down'
high_demand_mask = df['impressions_90d'] >= 500

print(f"Total stale pages (update >= {stale_threshold}d): {stale_mask.sum()}")
print(f"Of those stale pages, how many are actually declining: {((stale_mask) & (declining_mask)).sum()} ({((stale_mask) & (declining_mask)).sum() / stale_mask.sum() * 100:.1f}%)")
print(f"Of those stale pages, how many have high search demand: {((stale_mask) & (high_demand_mask)).sum()}")
print(f"Target priority pages (Stale AND Declining AND High Demand): {((stale_mask) & (declining_mask) & (high_demand_mask)).sum()}")


Total stale pages (update >= 180d): 174
Of those stale pages, how many are actually declining: 82 (47.1%)
Of those stale pages, how many have high search demand: 17
Target priority pages (Stale AND Declining AND High Demand): 16


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.